In [1]:
# imports

from dotenv import load_dotenv
from groq import Groq
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# The usual start

load_dotenv(override=True)
import os
groq_api_key = os.getenv('GROQ_API_KEY')
groq = Groq()

In [3]:
def push(message):
    print(f"Push: {message}")
    # Replace 'Harry' with your actual ntfy topic name
    ntfy_url = "https://ntfy.sh/Harry"
    
    # ntfy takes the message directly in the 'data' parameter
    requests.post(ntfy_url, data=message.encode('utf-8'))


In [4]:
def record_user_details(email, name="Name not provided", notes= "Notes not provided"):
    message = f"New Lead: {name} ({email})\nNotes: {notes}"
    
    push(message) # Call the push function to send the message to ntfy.sh
    
    return{"recorded" : "ok"}

In [5]:
def record_unknown_question(question):
    message = f"❓ Unknown Question: {question}"
    
    push(message) # Call the push function to send the message to ntfy.sh
    
    return{"recorded" : "ok"}

In [6]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [7]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [8]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    
    # Map tool names directly to their Python functions
    tool_mapping = {
        "record_user_details": record_user_details,
        "record_unknown_question": record_unknown_question
    }
    
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_id = tool_call.id
        arguments = json.loads(tool_call.function.arguments)
        
        print(f"Executing tool: {tool_name}", flush=True)
        
        # 1. Check if the tool exists in your mapping
        if tool_name in tool_mapping:
            try:
                # Execute the function with unpacked arguments
                execution_result = tool_mapping[tool_name](**arguments)
                
                # Ensure result is a string before passing to OpenAI
                if isinstance(execution_result, (dict, list)):
                    content_string = json.dumps(execution_result)
                else:
                    content_string = str(execution_result)
                    
            except Exception as e:
                # Catch internal function errors so the whole run doesn't crash
                content_string = json.dumps({"status": "error", "message": str(e)})
        else:
            # Handle cases where OpenAI invents a tool name (hallucination)
            content_string = json.dumps({"status": "error", "message": f"Tool {tool_name} not found"})
            
        # 2. Append standard OpenAI tool response format
        results.append({
            "role": "tool", 
            "tool_call_id": tool_id,
            "content": content_string 
        })
        
    return results


In [ ]:
# def handle_tool_calls(tool_calls):
#     results = []
#     for tool_call in tool_calls:
#         tool_name = tool_call.function.name
#         arguments = json.loads(tool_call.function.arguments)
#         print(f"Tool call: {tool_name} ", flush=True)
        
#         if tool_name == "record_user_details":
#             result = record_user_details(**arguments)
#         elif tool_name == "record_unknown_question":
#             result = record_unknown_question(**arguments)
            
#         results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id })
#     return results

In [18]:
reader = PdfReader("me/Harsh Asarsa.pdf")
linkedin = ""
github = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Harsh Asarsa"

In [19]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website."

system_prompt += f"If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career."
system_prompt += f"If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n## Github Profile:\n{github}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [20]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message} ]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json

        response = groq.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # If the LLM wants to call a tool, we do that!
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [21]:
gr.ChatInterface(chat, chatbot=gr.Chatbot()).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


c:\Users\dell\Documents\projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\dell\Documents\projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\dell\Documents\projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Executing tool: record_user_details
Push: New Lead: Malw (abc@gmail.com)
Notes: Notes not provided


c:\Users\dell\Documents\projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
